# Fondazione — Creazione degli indici

Questo notebook crea gli indici sulle tre collezioni (`film`, `persone`, `utenti`) e ne dimostra l'utilità.

Principio guida: **ogni indice esiste per accelerare una query concreta dell'applicazione.** Accanto a ciascun indice è indicata la query che lo giustifica, materiale diretto per la slide sulle scelte tecniche.

Prerequisito: MongoDB attivo su `localhost:27017` con il database `movie_database` popolato (collezioni `film` e `persone`).

In [ ]:
from connessione import get_db
from pymongo import ASCENDING, DESCENDING

db = get_db()
db.command("ping")
print("Connesso. Documenti per collezione:")
for nome in ["film", "persone", "utenti"]:
    print(f"  {nome}: {db[nome].count_documents({})}")

## Indici sulla collezione `film`

### 1. Indice su titolo
Serve la ricerca per titolo: match esatto `{ titolo: "Inception" }` e ricerca per prefisso con regex ancorata `{ titolo: /^Incep/ }` (che sfrutta l'indice). Un indice semplice, ordinato, è sufficiente per il caso attuale; per una ricerca full-text a parole servirebbe invece un indice testuale, che teniamo fuori finché non serve.

In [ ]:
db["film"].create_index([("titolo", ASCENDING)], name="idx_titolo")

### 2. Indice composto (genere + rating)
Serve due query in una: filtrare per genere `{ generi: "Azione" }` e la discovery "migliori di un genere" (filtro per genere, ordinati per voto decrescente).

Grazie alla **regola del prefisso**, un indice composto `(generi, rating.tmdb_media)` copre anche le query sul solo `generi`, quindi evitiamo un indice separato su `generi` (niente indici ridondanti da giustificare).

`generi` è un array: l'indice è quindi **multikey**, MongoDB indicizza ogni elemento dell'array.

In [ ]:
db["film"].create_index(
    [("generi", ASCENDING), ("rating.tmdb_media", DESCENDING)],
    name="idx_genere_rating",
)

### 3. Indice su anno di uscita
Serve i filtri e gli ordinamenti per anno: `{ anno_uscita: { $gte: 2000 } }`, oppure ordinare il catalogo dal più recente.

In [ ]:
db["film"].create_index([("anno_uscita", ASCENDING)], name="idx_anno")

### 4. Indice su rating
Serve la discovery "più votati in assoluto": ordinamento per `rating.tmdb_media` decrescente (di norma con una soglia minima di voti per togliere i film con pochissime valutazioni).

In [ ]:
db["film"].create_index([("rating.tmdb_media", DESCENDING)], name="idx_rating")

### 5. Indice sulle keyword
Serve il filtro tematico sui contenuti: `{ "keyword": "vendetta" }`, combinabile con gli altri filtri (genere, anni, voti) nella ricerca del catalogo.

`keyword` è un array, quindi l'indice è **multikey**: MongoDB indicizza ogni parola chiave dell'array, e il match trova i film che contengono quella keyword.

In [ ]:
db["film"].create_index([("keyword", ASCENDING)], name="idx_keyword")

## Indici sulla collezione `persone`

### 6. Indice su reparto principale
Serve a isolare attori e registi dalle comparse e dalla troupe tecnica: `{ reparto_principale: { $in: ["Acting", "Directing"] } }`.

### 7. Indice su nome
Serve la ricerca di una persona per nome: match esatto e per prefisso con regex ancorata `{ nome: /^Leonardo/ }`.

In [ ]:
db["persone"].create_index([("reparto_principale", ASCENDING)], name="idx_reparto")
db["persone"].create_index([("nome", ASCENDING)], name="idx_nome")

## Collezione `utenti`

La collezione ha un solo documento (uso personale). Le letture avvengono per `_id` (`"utente_1"`), che MongoDB indicizza **già in automatico**. Aggiungere altri indici qui darebbe costo di mantenimento senza alcun beneficio, quindi non ne creiamo. Riconoscere quando **non** serve un indice è parte della motivazione tecnica.

## Verifica: elenco degli indici creati

In [ ]:
for nome in ["film", "persone", "utenti"]:
    print(f"--- {nome} ---")
    for idx in db[nome].list_indexes():
        print("  ", idx["name"], "->", idx.get("key", {}))
    print()

## Prova che l'indice viene effettivamente usato

Con `.explain("executionStats")` guardiamo il piano di esecuzione di una query. Il campo da cercare è lo `stage`: **IXSCAN** significa che la query usa un indice, **COLLSCAN** che scorre tutta la collezione. Questa è la dimostrazione da mostrare all'esame.

In [ ]:
# query di esempio: migliori film di genere Azione, ordinati per voto
piano = db["film"].find(
    {"generi": "Azione"}
).sort("rating.tmdb_media", DESCENDING).explain()

stats = piano["executionStats"]
print("Documenti esaminati:", stats["totalDocsExamined"])
print("Documenti restituiti:", stats["nReturned"])
print("Millisecondi:", stats["executionTimeMillis"])
print("Piano vincente:", piano["queryPlanner"]["winningPlan"])

Se lo `stage` mostra `IXSCAN` con `indexName: "idx_genere_rating"`, l'indice composto sta lavorando: la query non scorre tutti i 2000 film ma salta direttamente a quelli di genere Azione già ordinati per voto.